# 🚀 Automatic CMT Hyperparameter Optimization
**Medical-Grade X-Ray Inpainting - Fully Automated**

Just run the cells - no widgets, no buttons, pure automation!

In [ ]:
# 1. Setup - Clone repo and install dependencies
!git clone https://github.com/C0d3Crush/arcade-xray-inpainting.git
%cd arcade-xray-inpainting
!pip install -q torch torchvision matplotlib pillow opencv-python scikit-image pycocotools pandas numpy scipy

print("✅ Setup complete - all dependencies installed")

In [ ]:
# 2. Mount Google Drive and Setup ARCADE Dataset
from google.colab import drive
drive.mount('/content/drive')

# Set ARCADE dataset path
ARCADE_ZIP = '/content/drive/MyDrive/arcade.zip'

# Extract ARCADE dataset
!unzip -q -o "$ARCADE_ZIP" -d /content/arcade-xray-inpainting/data/

# The zip contains 'arcade/stenosis/' but code expects 'arcade/syntax/'
# Create symlink to match expected path
!ln -sf /content/arcade-xray-inpainting/data/arcade/stenosis /content/arcade-xray-inpainting/data/arcade/syntax

# Verify dataset structure
!find data/arcade -name '*.json' | head -3
!ls -la data/arcade/

print("✅ ARCADE dataset ready")

In [ ]:
# 3. Automatic Hyperparameter Optimization - No Widgets!
import sys
import time
import json
import numpy as np
from datetime import datetime

# Add src to path
sys.path.append('src')

def run_automatic_optimization():
    """Fully automatic optimization - no user input needed"""
    
    print("🚀 AUTOMATIC HYPERPARAMETER OPTIMIZATION")
    print("🏥 Medical-Grade CMT Inpainting - Pure Automation")
    print("=" * 60)
    
    # Check GPU
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cuda':
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️ Using CPU")
    
    # Medical-aware quality scoring
    def evaluate_medical_quality(metrics):
        psnr = metrics.get('val_psnr', 0)
        ssim = metrics.get('val_ssim', 0)
        train_loss = metrics.get('train_loss', float('inf'))
        val_loss = metrics.get('val_loss', float('inf'))
        
        # Medical constraints
        if psnr > 70: return -float('inf'), "Overfitting detected"
        if train_loss < 0.05: return -float('inf'), "Trivial learning"
        if val_loss > train_loss * 3: return -float('inf'), "Severe overfitting"
        
        # Quality scoring (30-50 dB optimal)
        if 30 <= psnr <= 50:
            psnr_score = 1.0 - abs(psnr - 40) / 10
        else:
            psnr_score = 0.1
        
        ssim_score = min(ssim, 0.95)
        quality_score = psnr_score * 0.6 + ssim_score * 0.4
        
        return quality_score, "Medical grade" if quality_score > 0.6 else "Below standard"
    
    # Memory-optimized configurations for T4 GPU
    configs = [
        {'foreground_prob': 0.9, 'patches_per_image': 6, 'batch_size': 8, 'lr': 1e-4, 'max_shapes': 5},
        {'foreground_prob': 0.85, 'patches_per_image': 8, 'batch_size': 6, 'lr': 5e-5, 'max_shapes': 7},
        {'foreground_prob': 0.95, 'patches_per_image': 4, 'batch_size': 12, 'lr': 2e-4, 'max_shapes': 3},
        {'foreground_prob': 0.8, 'patches_per_image': 10, 'batch_size': 4, 'lr': 1e-4, 'max_shapes': 6},
        {'foreground_prob': 0.9, 'patches_per_image': 6, 'batch_size': 8, 'lr': 1.5e-4, 'max_shapes': 4}
    ]
    
    # Conservative memory filtering for T4
    if device == 'cuda':
        configs = [c for c in configs if c['patches_per_image'] * c['batch_size'] <= 60]  # Reduced limit
    else:
        configs = [c for c in configs if c['patches_per_image'] * c['batch_size'] <= 32]
    
    print(f"🎯 Testing {len(configs)} T4-optimized configurations")
    print(f"⏱️ Estimated time: {len(configs) * 2:.0f} minutes")
    print()
    
    results = []
    best_score = -float('inf')
    best_config = None
    
    # Import training function
    try:
        from train import train_model
        print("✅ Successfully imported train_model")
    except ImportError as e:
        print(f"❌ Import error: {e}")
        return None, []
    
    start_time = time.time()
    
    # Run experiments automatically
    for i, config in enumerate(configs, 1):
        print(f"🧪 Experiment {i}/{len(configs)}: fg={config['foreground_prob']:.2f}, ppi={config['patches_per_image']}, bs={config['batch_size']}, lr={config['lr']:.0e}")
        
        try:
            result = train_model(
                train_img='data/arcade/syntax/train/images',
                train_ann='data/arcade/syntax/train/annotations/train.json', 
                val_img='data/arcade/syntax/val/images',
                val_ann='data/arcade/syntax/val/annotations/val.json',
                epochs=2,  # Reduced for faster optimization
                batch_size=config['batch_size'],
                lr=config['lr'],
                input_size=64,
                device=device,
                output_dir=f'exp_{i:02d}',
                patch_mode=True,
                patches_per_image=config['patches_per_image'],
                foreground_prob=config['foreground_prob'],
                max_shapes=config['max_shapes'],
                smoke_test=True,
                smoke_size=12,  # Reduced for memory
                save_every=999
            )
            
            if result and result.get('final_metrics'):
                metrics = result['final_metrics']
                quality_score, quality_desc = evaluate_medical_quality(metrics)
                
                if quality_score > -float('inf'):
                    print(f"   ✅ PSNR: {metrics['val_psnr']:.1f} dB, SSIM: {metrics['val_ssim']:.3f}, Quality: {quality_score:.3f}")
                    
                    results.append({
                        'config': config,
                        'quality_score': quality_score,
                        'quality_desc': quality_desc,
                        **metrics
                    })
                    
                    if quality_score > best_score:
                        best_score = quality_score
                        best_config = config
                        print(f"   🏆 NEW BEST! Quality: {best_score:.3f}")
                else:
                    print(f"   ❌ {quality_desc}")
            else:
                print(f"   ❌ Training failed")
                
        except Exception as e:
            error_msg = str(e)
            if "CUDA out of memory" in error_msg:
                print(f"   ❌ GPU Memory Error - config too large for T4")
            else:
                print(f"   ❌ Error: {error_msg[:100]}...")
        
        print()
    
    # Results analysis
    total_time = time.time() - start_time
    
    print("🎉 OPTIMIZATION COMPLETE!")
    print("=" * 40)
    print(f"⏱️ Total time: {total_time/60:.1f} minutes")
    print(f"✅ Successful experiments: {len(results)}")
    
    if not results:
        print("❌ No successful experiments")
        print("💡 Try reducing batch sizes or patches_per_image for your GPU")
        return None, []  # Fixed: always return tuple
    
    # Sort by quality
    results.sort(key=lambda x: x['quality_score'], reverse=True)
    
    print("\n🏆 TOP 3 CONFIGURATIONS:")
    for i, result in enumerate(results[:3], 1):
        config = result['config']
        print(f"{i}. Quality: {result['quality_score']:.3f} | PSNR: {result['val_psnr']:.1f} dB | SSIM: {result['val_ssim']:.3f}")
        print(f"   fg_prob={config['foreground_prob']}, ppi={config['patches_per_image']}, bs={config['batch_size']}, lr={config['lr']:.0e}")
        print()
    
    # Medical analysis
    medical_grade = sum(1 for r in results if 30 <= r['val_psnr'] <= 50)
    print(f"🏥 Medical grade results: {medical_grade}/{len(results)} ({medical_grade/len(results)*100:.0f}%)")
    
    # Final recommendation  
    if best_config:
        print("\n🎯 OPTIMAL T4 CONFIGURATION:")
        for key, value in best_config.items():
            print(f"   --{key} {value}")
        
        print(f"\n🚀 PRODUCTION TRAINING COMMAND:")
        cmd = f"""python src/train.py \\
  --epochs 15 \\
  --patch_mode \\
  --input_size 64 \\
  --device {device} \\
  --foreground_prob {best_config['foreground_prob']} \\
  --patches_per_image {best_config['patches_per_image']} \\
  --batch_size {best_config['batch_size']} \\
  --lr {best_config['lr']:.0e} \\
  --max_shapes {best_config['max_shapes']}"""
        print(cmd)
        
        return best_config, results
    
    return None, []  # Fixed: always return tuple

# Run the automatic optimization
print("Starting automatic hyperparameter optimization...")
print("No widgets, no buttons - just pure automation! 🚀\n")

try:
    best_config, all_results = run_automatic_optimization()
    
    if best_config:
        print("\n✅ Optimization successful!")
        print("🎯 Optimal T4-compatible configuration found!")
    else:
        print("\n❌ Optimization failed - check setup")
        print("💡 Memory issues may require smaller batch sizes")
except Exception as e:
    print(f"\n❌ Optimization error: {str(e)}")
    print("💡 Try restarting runtime and running again")

In [ ]:
# 4. Optional: Run full training with optimal config
# Uncomment and run this cell after optimization completes

# if 'best_config' in globals() and best_config:
#     print("🚀 Starting production training with optimal configuration...")
#     
#     from train import train_model
#     
#     final_result = train_model(
#         train_img='data/arcade/syntax/train/images',
#         train_ann='data/arcade/syntax/train/annotations/train.json',
#         val_img='data/arcade/syntax/val/images', 
#         val_ann='data/arcade/syntax/val/annotations/val.json',
#         epochs=20,  # Full training
#         batch_size=best_config['batch_size'],
#         lr=best_config['lr'],
#         input_size=64,
#         device='cuda' if torch.cuda.is_available() else 'cpu',
#         output_dir='production_model',
#         patch_mode=True,
#         patches_per_image=best_config['patches_per_image'],
#         foreground_prob=best_config['foreground_prob'],
#         max_shapes=best_config['max_shapes']
#     )
#     
#     if final_result:
#         metrics = final_result['final_metrics']
#         print(f"\n🎉 Production training complete!")
#         print(f"📊 Final PSNR: {metrics['val_psnr']:.1f} dB")
#         print(f"📊 Final SSIM: {metrics['val_ssim']:.3f}")
#         print(f"💾 Model saved to: production_model/")
# else:
#     print("❌ No optimal config found - run optimization first")

print("💡 Uncomment the code above to run full production training")